# Step 1: The 2D Ising Model

This notebook is the foundation for the entire series. We build and simulate the two-dimensional Ising model from scratch — the same code your cohort produced in the mandatory project — and use it to observe a phase transition for the first time.

In the notebooks that follow, this code will be the starting point from which we extract critical exponents, introduce quantum spins, and eventually reach tensor networks. Make sure you are comfortable with every line here before moving on.

**What you will build:**
- A random spin lattice with periodic boundary conditions
- An energy function based on nearest-neighbour interactions
- A Metropolis Monte Carlo update rule
- A temperature sweep that reveals the ferromagnetic phase transition

**Prerequisites:** Basic Python and NumPy, thermodynamics at the level of partition functions and free energy.

## 1.1 The Hamiltonian

The Ising model assigns a spin $s_i \in \{+1, -1\}$ to each site $i$ of a lattice. The energy of a configuration is given by the Hamiltonian

$$H = -J \sum_{\langle i,j \rangle} s_i s_j$$

where the sum runs over all nearest-neighbour pairs $\langle i,j \rangle$ and $J > 0$ is the ferromagnetic coupling constant (aligned spins lower the energy).

On a 2D square lattice with periodic boundary conditions, each site has four nearest neighbours. The critical temperature in 2D is known exactly from Onsager's solution:

$$T_c = \frac{2J}{k_B \ln(1 + \sqrt{2})} \approx 2.269 \frac{J}{k_B}$$

We will work in units where $J = k_B = 1$, so $T_c \approx 2.269$.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

T_C_EXACT = 2.0 / np.log(1 + np.sqrt(2))  # Onsager exact result (J = k_B = 1)
print(f"Exact critical temperature: T_c = {T_C_EXACT:.4f}")

## 1.2 Initialising the Lattice

We start from a random configuration: each spin is independently drawn from $\{+1, -1\}$ with equal probability. This represents an infinite-temperature initial state with no correlations.

In [ ]:
def lattice_init(size):
    """
    Returns a (size x size) lattice of random spins in {+1, -1}.
    """
    return 2 * np.random.randint(0, 2, size=(size, size)) - 1

## 1.3 Energy of the Full Lattice

To compute the total energy we sum over all nearest-neighbour pairs. To avoid double-counting, we only count each pair once: for each site $(i, j)$ we include the right neighbour $(i, j+1)$ and the bottom neighbour $(i+1, j)$. Periodic boundary conditions are imposed with the modulo operator.

In [ ]:
def total_energy(lattice, size, J):
    """
    Total energy of the lattice under H = -J sum_{<ij>} s_i s_j.
    Each pair counted once (right + down neighbours only).
    """
    energy = 0
    for i in range(size):
        for j in range(size):
            spin = lattice[i, j]
            right_nn = lattice[i, (j + 1) % size]
            down_nn  = lattice[(i + 1) % size, j]
            energy += -J * spin * right_nn
            energy += -J * spin * down_nn
    return energy

## 1.4 Local Energy Change for a Single Spin Flip

Recalculating the total energy after every proposed move would cost $O(N)$ per step. We can do better: flipping spin $s_i$ only changes the interactions between site $i$ and its four nearest neighbours. The energy change is

$$\Delta E = 2 J s_i \sum_{\delta} s_{i+\delta}$$

where the sum is over the four neighbours. This is $O(1)$ regardless of system size.

In [ ]:
def delta_energy(lattice, size, i, j, J):
    """
    Energy change from flipping spin at (i, j).
    Uses only the four nearest neighbours — O(1) per step.
    """
    spin   = lattice[i, j]
    nn_sum = (lattice[i, (j + 1) % size] +
              lattice[i, (j - 1) % size] +
              lattice[(i - 1) % size, j] +
              lattice[(i + 1) % size, j])
    # energy_after - energy_before = (-J)(-spin)(nn_sum) - (-J)(spin)(nn_sum)
    return 2 * J * spin * nn_sum


def magnetization(lattice):
    """Mean magnetization per spin."""
    return np.mean(lattice)

## 1.5 The Metropolis Algorithm

The Metropolis algorithm samples configurations from the Boltzmann distribution $P \propto e^{-\beta H}$ by a random walk through configuration space. At each step:

1. Choose a site $(i, j)$ uniformly at random.
2. Compute $\Delta E$ for flipping that spin.
3. Accept the flip with probability $\min(1,\, e^{-\beta \Delta E})$.

If $\Delta E \leq 0$ the flip is always accepted (it lowers the energy). If $\Delta E > 0$ it is accepted with a Boltzmann factor that suppresses energetically costly moves at low temperature.

We pre-generate the random numbers to avoid calling `np.random` inside the hot loop.

In [ ]:
def metropolis(lattice, size, beta, J, steps):
    """
    Evolve the lattice in-place for `steps` Metropolis moves.
    """
    rand_i      = np.random.randint(0, size, size=steps)
    rand_j      = np.random.randint(0, size, size=steps)
    rand_accept = np.random.random(size=steps)

    for step in range(steps):
        i, j   = rand_i[step], rand_j[step]
        dE     = delta_energy(lattice, size, i, j, J)
        if dE < 0 or np.exp(-beta * dE) > rand_accept[step]:
            lattice[i, j] *= -1

## 1.6 Running the Simulation

We evolve the lattice in batches of `batch_size` steps and record the magnetization after each batch. This gives us a time series to check for equilibration.

In [ ]:
def run_simulation(size, beta, total_steps, J, batch_size):
    """
    Run the Ising model simulation.

    Returns
    -------
    starting_lattice   : initial random configuration
    final_lattice      : configuration after all steps
    mag_history        : magnetization sampled after each batch
    """
    lattice          = lattice_init(size)
    starting_lattice = lattice.copy()
    n_batches        = total_steps // batch_size
    mag_history      = np.zeros(n_batches)

    for batch in range(n_batches):
        metropolis(lattice, size, beta, J, steps=batch_size)
        mag_history[batch] = magnetization(lattice)

    return starting_lattice, lattice, mag_history

In [ ]:
# Parameters
SIZE       = 25
BATCH_SIZE = 1000
STEPS      = 1_000_000
T          = 1.0          # well below T_c — expect ordered state
J          = 1.0
beta       = 1.0 / T

start, final, mag_hist = run_simulation(SIZE, beta, STEPS, J, BATCH_SIZE)

E_start = total_energy(start, SIZE, J)
E_final = total_energy(final, SIZE, J)

steps_axis = np.arange(BATCH_SIZE, STEPS + BATCH_SIZE, BATCH_SIZE)

fig = plt.figure(figsize=(10, 8))
ax1 = plt.subplot2grid((2, 2), (0, 0))
ax2 = plt.subplot2grid((2, 2), (0, 1))
ax3 = plt.subplot2grid((2, 2), (1, 0), colspan=2)

im1 = ax1.imshow(start, cmap='RdBu', vmin=-1, vmax=1)
ax1.set_title(f"a) Initial lattice  E = {E_start:.0f}", fontsize=13)
ax1.set_xlabel("j (column)", fontsize=12)
ax1.set_ylabel("i (row)", fontsize=12)
plt.colorbar(im1, ax=ax1, label="spin")

im2 = ax2.imshow(final, cmap='RdBu', vmin=-1, vmax=1)
ax2.set_title(f"b) Final lattice  E = {E_final:.0f}", fontsize=13)
ax2.set_xlabel("j (column)", fontsize=12)
plt.colorbar(im2, ax=ax2, label="spin")

ax3.plot(steps_axis, mag_hist)
ax3.axhline(0, lw=1, ls='--', color='k', alpha=0.5)
ax3.set_xlabel("Monte Carlo steps", fontsize=12)
ax3.set_ylabel("Magnetization $m$", fontsize=12)
ax3.set_title("c) Magnetization vs MC time", fontsize=13)

plt.tight_layout()
plt.show()

## 1.7 The Phase Transition: Temperature Sweep

At $T \ll T_c$ the system orders ferromagnetically: $\langle |m| \rangle \to 1$. At $T \gg T_c$ thermal fluctuations destroy order: $\langle |m| \rangle \to 0$. The transition between these two phases is the ferromagnetic phase transition.

We sweep temperature through the critical region $T \in [2.0, 3.0]$ and record the equilibrium magnetization at each temperature. We use the final lattice from one temperature as the starting point for the next (a "heating" protocol).

In [ ]:
def temperature_sweep(size, T_values, steps_per_T, J, batch_size, n_average_batches):
    """
    Sweep through temperatures and record equilibrium |m| at each.

    Parameters
    ----------
    n_average_batches : number of final batches used to average the magnetization

    Returns
    -------
    magnetizations : array of <|m|> at each temperature
    lattice_states : snapshot of lattice at each temperature
    """
    lattice        = lattice_init(size)
    n_batches      = steps_per_T // batch_size
    magnetizations = np.zeros(len(T_values))
    lattice_states = np.empty(len(T_values), dtype=object)

    for k, T in enumerate(T_values):
        beta    = 1.0 / T
        mag_buf = np.zeros(n_batches)

        for batch in range(n_batches):
            metropolis(lattice, size, beta, J, steps=batch_size)
            mag_buf[batch] = magnetization(lattice)

        magnetizations[k] = np.abs(np.mean(mag_buf[-n_average_batches:]))
        lattice_states[k] = lattice.copy()
        print(f"T = {T:.2f}   <|m|> = {magnetizations[k]:.4f}")

    return magnetizations, lattice_states

In [ ]:
T_VALUES = np.arange(2.0, 3.1, 0.1)

magnetizations, lattice_states = temperature_sweep(
    size=25,
    T_values=T_VALUES,
    steps_per_T=1_000_000,
    J=1.0,
    batch_size=100_000,
    n_average_batches=1
)

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(T_VALUES, magnetizations, 'o-', lw=2, ms=7)
plt.axvline(T_C_EXACT, color='r', ls='--', lw=1.5, label=f'Exact $T_c = {T_C_EXACT:.3f}$')
plt.axhline(0, color='k', ls='--', lw=1, alpha=0.4)
plt.xlabel('Temperature $T$  ($J/k_B$)', fontsize=13)
plt.ylabel('$\\langle |m| \\rangle$', fontsize=13)
plt.title('Order parameter across the ferromagnetic phase transition', fontsize=13)
plt.legend(fontsize=12)
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Lattice snapshots across the sweep
indices = [0, 1, 3, 4, 9, 10]
labels  = list('abcdef')

fig, axes = plt.subplots(2, 3, figsize=(13, 9))
for ax, k, lbl in zip(axes.flat, indices, labels):
    im = ax.imshow(lattice_states[k], cmap='RdBu', vmin=-1, vmax=1)
    ax.set_title(f"{lbl}) $T = {T_VALUES[k]:.2f}$", fontsize=14)
    ax.set_xlabel('j', fontsize=11)
    ax.set_ylabel('i', fontsize=11)
    plt.colorbar(im, ax=ax, fraction=0.045, pad=0.05)

plt.suptitle('Spin configurations across the phase transition', fontsize=15, y=1.01)
plt.tight_layout()
plt.show()

## 1.8 What We Observe — and What We Are Missing

The magnetization curve drops sharply near $T_c \approx 2.27$, but it does not go discontinuously to zero. This is a continuous (second-order) phase transition. Close to $T_c$, the order parameter vanishes as a power law:

$$\langle |m| \rangle \sim (T_c - T)^\beta$$

where $\beta = 1/8$ in 2D (not to be confused with inverse temperature). The rounding and shifting of the transition you see in the plot is a **finite-size effect** — a 25×25 lattice is small. In the next notebook, we will use systems of different sizes to extrapolate to the thermodynamic limit and extract the critical exponent $\beta$ directly.

---

## Exercises

1. **Antiferromagnet.** Set $J = -1$. What changes in the magnetization curve? What order parameter would you need to detect the transition?

2. **System size.** Rerun the temperature sweep with `size = 10` and `size = 50`. How does the sharpness of the transition change? Where does the apparent $T_c$ shift?

3. **Equilibration.** At $T = 2.3$ (just above $T_c$), plot the full magnetization time series. How many steps does the system take to equilibrate from a random initial state?

4. **Cooling vs heating.** Run the sweep in the reverse direction ($T$ from 3.0 down to 2.0) starting from the disordered state. Does the magnetization curve change? What is the physical meaning of any difference?